## 1. import

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool

## 2. config

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }

## 3. column groups

In [3]:
# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE

LOG_SOURCE_COLS = [
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

## 4. utility

In [4]:
# =========================
# Utilities
# =========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def cast_dtypes(train_df, test_df):
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    return train_df, test_df

## 5. load data

In [5]:
seed_everything(CFG.SEED)

train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)

train_df, test_df = cast_dtypes(train_df, test_df)

print(train_df.shape, test_df.shape)

(630000, 21) (270000, 20)


## 6. label encode

In [6]:
le = LabelEncoder()
y = le.fit_transform(train_df[CFG.TARGET_COL])
class_names = list(le.classes_)
n_classes = len(class_names)

print("classes:", class_names)

classes: ['High', 'Low', 'Medium']


## 7. LOG features

In [7]:
def add_log_features(df: pd.DataFrame, source_cols):
    df = df.copy()
    created_cols = []

    for col in source_cols:
        x = pd.to_numeric(df[col], errors="coerce").astype(float)

        x_nonneg = x.where(x >= 0, np.nan)

        log1p_x = np.log1p(x_nonneg)
        sqrt_x = np.sqrt(x_nonneg)
        is_zero = (x_nonneg == 0).astype(float)
        diff_log1p = x_nonneg - log1p_x

        new_map = {
            f"log__{col}__log1p": log1p_x,
            f"log__{col}__sqrt": sqrt_x,
            f"log__{col}__is_zero": is_zero,
            f"log__{col}__diff_log1p": diff_log1p,
        }

        for new_col, values in new_map.items():
            df[new_col] = values
            created_cols.append(new_col)

    return df, created_cols


train_df, log_cols = add_log_features(train_df, LOG_SOURCE_COLS)
test_df, _ = add_log_features(test_df, LOG_SOURCE_COLS)

FEATURE_COLS_LOG = FEATURE_COLS_BASE + log_cols

print("LOG_SOURCE_COLS:", LOG_SOURCE_COLS)
print("n log cols:", len(log_cols))
print("log cols sample:", log_cols[:10])

LOG_SOURCE_COLS: ['Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
n log cols: 32
log cols sample: ['log__Soil_Moisture__log1p', 'log__Soil_Moisture__sqrt', 'log__Soil_Moisture__is_zero', 'log__Soil_Moisture__diff_log1p', 'log__Organic_Carbon__log1p', 'log__Organic_Carbon__sqrt', 'log__Organic_Carbon__is_zero', 'log__Organic_Carbon__diff_log1p', 'log__Electrical_Conductivity__log1p', 'log__Electrical_Conductivity__sqrt']


## 8. CV train

In [8]:
X = train_df[FEATURE_COLS_LOG].copy()
X_test = test_df[FEATURE_COLS_LOG].copy()

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 60)
    print(f"fold {fold}")

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_test,
        cat_features=CAT_ALL_BASE,
    )

    model = CatBoostClassifier(**CFG.MODEL_PARAMS)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba += te_proba / CFG.N_SPLITS

    va_pred = np.argmax(va_proba, axis=1)
    fold_bacc = balanced_accuracy_score(y_va, va_pred)
    fold_scores.append(fold_bacc)

    print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

fold 1
0:	learn: 1.0374400	test: 1.0373987	best: 1.0373987 (0)	total: 113ms	remaining: 5m 38s
200:	learn: 0.0642986	test: 0.0645448	best: 0.0645448 (200)	total: 3.22s	remaining: 44.9s
400:	learn: 0.0627890	test: 0.0634439	best: 0.0634439 (400)	total: 5.95s	remaining: 38.6s
600:	learn: 0.0611927	test: 0.0624727	best: 0.0624727 (600)	total: 8.67s	remaining: 34.6s
800:	learn: 0.0598135	test: 0.0617902	best: 0.0617902 (800)	total: 11.5s	remaining: 31.6s
1000:	learn: 0.0587765	test: 0.0613288	best: 0.0613282 (996)	total: 14.2s	remaining: 28.4s
1200:	learn: 0.0579249	test: 0.0610090	best: 0.0610090 (1200)	total: 16.8s	remaining: 25.2s
1400:	learn: 0.0569998	test: 0.0607335	best: 0.0607335 (1400)	total: 19.6s	remaining: 22.3s
1600:	learn: 0.0562695	test: 0.0605145	best: 0.0605144 (1596)	total: 22.2s	remaining: 19.4s
1800:	learn: 0.0555434	test: 0.0603093	best: 0.0603093 (1800)	total: 24.9s	remaining: 16.6s
2000:	learn: 0.0548049	test: 0.0601506	best: 0.0601506 (2000)	total: 27.7s	remaining: 1

## 9. bias search helpers

In [9]:
def predict_with_bias(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    adjusted = proba + bias.reshape(1, -1)
    return np.argmax(adjusted, axis=1)


def evaluate_per_class_recall(y_true: np.ndarray, y_pred: np.ndarray, class_names=None):
    n_classes = len(np.unique(y_true))
    recalls = {}
    for cls_idx in range(n_classes):
        mask = (y_true == cls_idx)
        recall = float((y_pred[mask] == y_true[mask]).mean()) if mask.sum() > 0 else np.nan
        key = class_names[cls_idx] if class_names is not None else cls_idx
        recalls[key] = recall
    return recalls


def search_topk_class_bias(
    oof_proba: np.ndarray,
    y_true: np.ndarray,
    bias_grid=None,
    fix_first_class_zero=True,
    top_k=10,
    verbose=True,
):
    n_classes = oof_proba.shape[1]

    if bias_grid is None:
        bias_grid = [-0.60, -0.50, -0.40, -0.30, -0.20, -0.10, -0.05,
                     0.0,
                     0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

    results = []

    if fix_first_class_zero:
        for rest in product(bias_grid, repeat=n_classes - 1):
            bias = np.array([0.0] + list(rest), dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)
            results.append((score, bias.copy()))
    else:
        for bias_tuple in product(bias_grid, repeat=n_classes):
            bias = np.array(bias_tuple, dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)
            results.append((score, bias.copy()))

    results.sort(key=lambda x: x[0], reverse=True)
    top_results = results[:top_k]

    if verbose:
        print("Top coarse candidates:")
        for i, (score, bias) in enumerate(top_results, 1):
            print(f"{i:2d}: score={score:.6f}, bias={bias}")

    return top_results


def refine_from_topk_candidates(
    oof_proba: np.ndarray,
    y_true: np.ndarray,
    top_results,
    step=0.01,
    radius=0.08,
    fix_first_class_zero=True,
    verbose=True,
):
    n_classes = oof_proba.shape[1]
    best_score = -1.0
    best_bias = None

    for coarse_score, coarse_bias in top_results:
        grids = []
        for i in range(n_classes):
            if fix_first_class_zero and i == 0:
                grids.append([0.0])
            else:
                center = coarse_bias[i]
                vals = np.arange(center - radius, center + radius + 1e-12, step)
                vals = np.round(vals, 6)
                grids.append(vals.tolist())

        for bias_tuple in product(*grids):
            bias = np.array(bias_tuple, dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)

            if score > best_score:
                best_score = score
                best_bias = bias.copy()

    if verbose:
        print("Refined best:")
        print(f"best_score={best_score:.6f}")
        print(f"best_bias ={best_bias}")

    return best_bias, best_score

## 10. bias optimize + summary

In [10]:
print("=" * 60)
print("Bias optimization (wide search)")

print("fold scores:", [round(s, 6) for s in fold_scores])
print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

oof_pred_base = np.argmax(oof_proba, axis=1)
oof_bacc_base = balanced_accuracy_score(y, oof_pred_base)
print(f"OOF balanced_accuracy (no bias) = {oof_bacc_base:.6f}")

recalls_base = evaluate_per_class_recall(y, oof_pred_base, class_names)
print("Per-class recall (no bias):")
for k, v in recalls_base.items():
    print(f"  {k}: {v:.6f}")

coarse_grid = [
    -0.60, -0.50, -0.40, -0.30, -0.20, -0.10, -0.05,
     0.00,
     0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60
]

top_results = search_topk_class_bias(
    oof_proba=oof_proba,
    y_true=y,
    bias_grid=coarse_grid,
    fix_first_class_zero=True,
    top_k=12,
    verbose=True,
)

best_bias, best_bias_score = refine_from_topk_candidates(
    oof_proba=oof_proba,
    y_true=y,
    top_results=top_results,
    step=0.01,
    radius=0.08,
    fix_first_class_zero=True,
    verbose=True,
)

oof_pred_bias = predict_with_bias(oof_proba, best_bias)
oof_bacc_bias = balanced_accuracy_score(y, oof_pred_bias)
print(f"\nOOF balanced_accuracy (with bias) = {oof_bacc_bias:.6f}")

recalls_bias = evaluate_per_class_recall(y, oof_pred_bias, class_names)
print("Per-class recall (with bias):")
for k, v in recalls_bias.items():
    print(f"  {k}: {v:.6f}")

print("\nDelta:")
print(f"OOF gain = {oof_bacc_bias - oof_bacc_base:+.6f}")
for cls in class_names:
    print(f"{cls} recall gain = {recalls_bias[cls] - recalls_base[cls]:+.6f}")

Bias optimization (wide search)
fold scores: [np.float64(0.957408), np.float64(0.961315), np.float64(0.95949), np.float64(0.958643), np.float64(0.957761)]
cv mean balanced_accuracy = 0.958924
cv std  balanced_accuracy = 0.001397
OOF balanced_accuracy (no bias) = 0.958924
Per-class recall (no bias):
  High: 0.909087
  Low: 0.994274
  Medium: 0.973410
Top coarse candidates:
 1: score=0.965139, bias=[ 0.  -0.4 -0.6]
 2: score=0.965042, bias=[ 0.  -0.3 -0.6]
 3: score=0.964924, bias=[ 0.  -0.5 -0.6]
 4: score=0.964903, bias=[ 0.  -0.2 -0.6]
 5: score=0.964756, bias=[ 0.  -0.1 -0.6]
 6: score=0.964692, bias=[ 0.  -0.6 -0.6]
 7: score=0.964630, bias=[ 0.   -0.05 -0.6 ]
 8: score=0.964490, bias=[ 0.   0.  -0.6]
 9: score=0.964288, bias=[ 0.    0.05 -0.6 ]
10: score=0.964007, bias=[ 0.   0.1 -0.6]
11: score=0.963729, bias=[ 0.  -0.5 -0.5]
12: score=0.963718, bias=[ 0.  -0.4 -0.5]
Refined best:
best_score=0.965998
best_bias =[ 0.   -0.31 -0.68]

OOF balanced_accuracy (with bias) = 0.965998
Per-

## 11. save

In [11]:
np.save("best_class_bias_log_min.npy", best_bias)

test_pred_bias = predict_with_bias(test_proba, best_bias)
test_label_bias = le.inverse_transform(test_pred_bias)

sub_bias_df = pd.DataFrame({
    CFG.ID_COL: test_df[CFG.ID_COL],
    CFG.TARGET_COL: test_label_bias
})

np.save("oof_catboost_log_min_bias_proba.npy", oof_proba)
np.save("pred_catboost_log_min_bias_proba.npy", test_proba)
sub_bias_df.to_csv("submission_catboost_log_min_bias.csv", index=False)

print("\nSaved:")
print("- best_class_bias_log_min.npy")
print("- oof_catboost_log_min_bias_proba.npy")
print("- pred_catboost_log_min_bias_proba.npy")
print("- submission_catboost_log_min_bias.csv")


Saved:
- best_class_bias_log_min.npy
- oof_catboost_log_min_bias_proba.npy
- pred_catboost_log_min_bias_proba.npy
- submission_catboost_log_min_bias.csv


## 12. bias confirm

In [12]:
loaded_bias = np.load("/kaggle/working/best_class_bias_log_min.npy")
print("le.classes_:", le.classes_)
print("best_bias  :", loaded_bias)

for cls, bias in zip(le.classes_, loaded_bias):
    print(f"{cls}: {bias:.6f}")

le.classes_: ['High' 'Low' 'Medium']
best_bias  : [ 0.   -0.31 -0.68]
High: 0.000000
Low: -0.310000
Medium: -0.680000
